# 22 — AI Project Manager

Feed it project updates — emails, status call notes, Slack threads — and it maintains a structured `ProjectState` with milestones, risks, blockers, and a green/amber/red RAG status.

In [ ]:
!pip install -q langchain-openai langchain-core python-dotenv

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class Milestone(BaseModel):
    name: str = Field(description='Milestone name.')
    due_date: str = Field(description='Target date, e.g. Q3-2024.')
    status: Literal['on_track', 'at_risk', 'delayed', 'complete'] = Field(description='Status.')

class Risk(BaseModel):
    description: str = Field(description='Risk description.')
    severity: Literal['low', 'medium', 'high', 'critical'] = Field(description='Severity.')
    owner: str = Field(description='Mitigation owner.')

class Blocker(BaseModel):
    description: str = Field(description='What is blocking progress.')
    raised_by: str = Field(description='Who raised it.')
    resolution_needed_from: str = Field(description='Who needs to act.')

class DeliverableOwner(BaseModel):
    deliverable: str = Field(description='Deliverable name.')
    owner: str = Field(description='Responsible person.')
    due_date: str = Field(description='Target date.')

class ProjectState(BaseModel):
    project_name: str = Field(description='Project name.')
    overall_status: Literal['green', 'amber', 'red'] = Field(description='RAG status.')
    milestones: list[Milestone] = Field(description='Milestones.')
    risks: list[Risk] = Field(description='Risks.')
    blockers: list[Blocker] = Field(description='Blockers.')
    deliverable_owners: list[DeliverableOwner] = Field(description='Ownership.')
    summary: str = Field(description='Executive summary.')

class UpdateInput(BaseModel):
    source: str = Field(description='Update source.')
    content: str = Field(description='Raw update text.')

In [ ]:
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI

_EXTRACT = SystemMessage('Extract only what this update mentions into a ProjectState delta. Leave other fields empty.')
_MERGE = SystemMessage('Merge the delta into the current state. Update existing items, remove resolved blockers, re-derive RAG status, write a fresh executive summary.')

def init_state(name):
    return ProjectState(project_name=name, overall_status='green', milestones=[], risks=[], blockers=[], deliverable_owners=[], summary='Project initialised.')

def apply_update(current, update):
    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0).with_structured_output(ProjectState)
    delta = llm.invoke([_EXTRACT, {'role': 'user', 'content': f'Project: {current.project_name}\nSource: {update.source}\nUpdate:\n{update.content}'}])
    return llm.invoke([_MERGE, {'role': 'user', 'content': f'Current:\n{current.model_dump_json(indent=2)}\n\nDelta ({update.source}):\n{delta.model_dump_json(indent=2)}'}])

def run(project_name, updates):
    state = init_state(project_name)
    history = [state]
    for u in updates:
        state = apply_update(state, u)
        history.append(state)
    return history

In [ ]:
updates = [
    UpdateInput(source='Kick-off (Week 1)', content='Three milestones: data migration Q3-2024, UAT Q4-2024, go-live Q1-2025. Sarah owns data migration, James owns UAT.'),
    UpdateInput(source='Status call (Week 4)', content='Data migration at risk. Vendor has not delivered API spec. Blocker for Sarah. Procurement must escalate.'),
    UpdateInput(source='Email (Week 6)', content='API spec received. Blocker resolved. But migration slipped 3 weeks to early Q4. UAT pushed to late Q4. Overall status amber.'),
]

history = run('Platform Migration -- Phase 2', updates)
for i, state in enumerate(history):
    label = 'INITIAL' if i == 0 else f'After update {i}'
    print(f'\n{"="*50}')
    print(f'{label} | Status: {state.overall_status.upper()}')
    print(f'Summary: {state.summary}')
    for m in state.milestones:
        print(f'  Milestone [{m.status}] {m.name} -- {m.due_date}')

## Exercise

Add a fourth update that marks the go-live milestone as complete and the overall project as green. Watch the state converge back to healthy.